# Smart Classroom AI - Train From Scratch

This notebook trains a custom CNN to classify classrooms as LOW, MEDIUM, or HIGH occupancy.
**No YOLO, No pre-trained weights** - we build everything ourselves!

**Before running: Go to Runtime -> Change Runtime Type -> T4 GPU (free)**

| Step | What We Do | Real Life Analogy |
|------|-----------|------------------|
| 1 | Import tools | Open your toolbox |
| 2 | Upload videos & extract frames | Show photos to a student |
| 3 | Prepare images | Resize all photos to same size |
| 4 | Build CNN brain | Draw a brain on paper |
| 5 | Train the model | Make the student study 30 times |
| 6 | Draw training charts | See how well the student improved |
| 7 | Confusion matrix | Exam report card |
| 8 | Export ONNX model | Save knowledge as a file |

In [ ]:
# STEP 1: Import all tools we need
# Think of this like opening your toolbox before starting work

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import os, zipfile, json
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from PIL import Image

# Use GPU if available (makes training ~10x faster for free!)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} - Training will be fast!')
else:
    print('No GPU - please enable T4 GPU in Runtime settings!')
print('All tools loaded!')

## STEP 2: Upload Your Classroom Videos

A video is just many photos shown quickly (like a flipbook).
We extract 1 photo every 10 frames to build our training dataset.

**What you need:** Record at least 2 videos per class:
- **LOW**: Classroom with 1-2 people
- **MEDIUM**: Classroom with 3-9 people  
- **HIGH**: Classroom with 10+ people

**Privacy tip**: Camera from behind/above - no faces visible!

In [ ]:
import cv2
from google.colab import files

def extract_frames(video_path, output_dir, label, every_n=10):
    """Extract one frame every N frames from a video and save as images"""
    save_dir = os.path.join(output_dir, label)
    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f'Video: {os.path.basename(video_path)}')
    print(f'  Total frames: {total}, FPS: {fps:.1f}')
    count, saved = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % every_n == 0:
            frame = cv2.resize(frame, (224, 224))
            out_path = os.path.join(save_dir, f'{label}_{saved:05d}.jpg')
            cv2.imwrite(out_path, frame)
            saved += 1
        count += 1
    cap.release()
    print(f'  Saved {saved} frames to {save_dir}/')
    return saved

# Upload your video files
print('Upload your classroom video files (.mp4 recommended)...')
uploaded = files.upload()

os.makedirs('dataset', exist_ok=True)
dataset_path = 'dataset'

for video_file in uploaded.keys():
    print(f'\nWhat label is: {video_file}')
    print('  Enter  low   for 1-2 people in classroom')
    print('  Enter  medium  for 3-9 people in classroom')
    print('  Enter  high  for 10 or more people in classroom')
    label = input('Your label: ').strip().lower()
    if label in ['low', 'medium', 'high']:
        extract_frames(video_file, dataset_path, label)
    else:
        print(f'Unknown label: {label} - skipping')

# Show summary
print('\n--- Dataset Summary ---')
total_imgs = 0
for cls in ['low', 'medium', 'high']:
    p = os.path.join(dataset_path, cls)
    n = len([f for f in os.listdir(p) if f.endswith('.jpg')]) if os.path.exists(p) else 0
    total_imgs += n
    status = 'OK' if n >= 100 else 'WARNING: try to get at least 100 images'
    print(f'  {cls.upper():8}: {n:4d} images  [{status}]')
print(f'  TOTAL   : {total_imgs} images')

## STEP 3: Prepare Images for Training

AI needs images in exactly the right format - like a picky eater!

**Data Augmentation** = creating MORE photos from your existing ones:
- Flip left-right (classroom is still a classroom when mirrored)
- Rotate slightly (camera might be tilted)
- Change brightness (different times of day)

200 photos with augmentation effectively becomes 2000+ examples!

In [ ]:
# Image size: 224x224 pixels is standard for CNNs
IMAGE_SIZE = 224
BATCH_SIZE = 32  # Process 32 images at a time

# Training transforms: WITH augmentation (artificial variety)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),         # Flip 50% of images
    transforms.RandomRotation(15),                   # Tilt up to 15 degrees
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),  # Lighting changes
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),  # Zoom in a bit
    transforms.ToTensor(),                           # Convert to numbers
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Standardize
])

# Validation transforms: NO augmentation (fair testing)
val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load images from dataset/ folder automatically
full_dataset = datasets.ImageFolder(root=dataset_path, transform=train_transforms)
class_names = full_dataset.classes  # ['high', 'low', 'medium'] - alphabetical order

print(f'Classes found: {class_names}')
print(f'Total images loaded: {len(full_dataset)}')

# Split: 80% for studying, 20% for final exam
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
val_ds.dataset.transform = val_transforms  # No augmentation on exam data

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Training images:   {train_size}')
print(f'Validation images: {val_size}')
print('Ready to train!')

## STEP 4: Build Our AI Brain From Scratch

A CNN (Convolutional Neural Network) works like eyes + brain:
```
Photo --> [Eye 1: sees edges] --> [Eye 2: sees shapes]
      --> [Eye 3: sees objects] --> [Eye 4: sees crowds]
      --> [Brain: decides LOW/MEDIUM/HIGH]
```

Every number (parameter) in this model **starts at zero** and is learned only from YOUR videos!

In [ ]:
class ClassroomCNN(nn.Module):
    """
    Our custom AI brain, built from scratch.
    Input:  224x224 color image
    Output: 3 scores (one for LOW, MEDIUM, HIGH)
    """
    def __init__(self, num_classes=3):
        super().__init__()

        # EYES: Scan the image for patterns
        self.features = nn.Sequential(
            # Eye Layer 1: Detect basic edges  (224->112 pixels)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),  # Keep training stable
            nn.ReLU(),           # Only pass positive signals
            nn.MaxPool2d(2, 2),  # Shrink image, keep strongest signals
            nn.Dropout2d(0.1),   # Randomly disable 10% to avoid memorizing

            # Eye Layer 2: Detect shapes  (112->56 pixels)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),

            # Eye Layer 3: Detect objects like chairs, people  (56->28 pixels)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            # Eye Layer 4: Detect crowd density patterns  (28->14 pixels)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),
        )

        # Summarize everything into 1 number per eye filter
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # BRAIN: Make final decision
        self.brain = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.5),  # Drop 50% during training - prevents cheating!
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)  # Final answer: score for each class
        )

    def forward(self, x):
        x = self.features(x)  # Look at image
        x = self.gap(x)       # Summarize
        x = self.brain(x)     # Decide
        return x

# Create model and move to GPU
model = ClassroomCNN(num_classes=len(class_names)).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model built from scratch!')
print(f'Total parameters: {total_params:,}')
print(f'All {total_params:,} parameters start at random values and will be')
print(f'learned ONLY from your {len(full_dataset)} classroom images!')

## STEP 5: Train the Model (Teach the AI)

This is the main learning step! Here's what happens each **epoch** (study session):
1. Show AI all training photos one batch at a time
2. AI makes a guess for each photo
3. We tell AI if it was right or wrong (calculate **loss**)
4. AI adjusts its brain to do better next time
5. After all photos, test on validation photos (no peeking!)

We repeat this **30 times**. Each time the AI should get a little smarter!

In [ ]:
criterion = nn.CrossEntropyLoss()  # Measures how wrong the AI is
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # Fixes mistakes
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # Slow down gradually

NUM_EPOCHS = 30
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_acc = 0.0
best_state = None

print(f'Starting {NUM_EPOCHS} training epochs...')
print('Watch the accuracy go UP and the loss go DOWN each epoch!')
print('=' * 70)

for epoch in range(NUM_EPOCHS):

    # --- TRAINING PHASE: AI is in learning mode ---
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0

    for batch_imgs, batch_labels in train_loader:
        batch_imgs = batch_imgs.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()              # Clear old corrections
        outputs = model(batch_imgs)        # AI looks at images and guesses
        loss = criterion(outputs, batch_labels)  # How wrong was the guess?
        loss.backward()                    # Figure out what caused the error
        optimizer.step()                   # Fix the brain

        t_loss += loss.item()
        t_correct += (outputs.argmax(1) == batch_labels).sum().item()
        t_total += batch_labels.size(0)

    # --- VALIDATION PHASE: AI is in exam mode (no learning) ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0

    with torch.no_grad():  # Don't adjust during exam!
        for val_imgs, val_labels in val_loader:
            val_imgs = val_imgs.to(device)
            val_labels = val_labels.to(device)
            val_out = model(val_imgs)
            v_loss += criterion(val_out, val_labels).item()
            v_correct += (val_out.argmax(1) == val_labels).sum().item()
            v_total += val_labels.size(0)

    # Calculate epoch results
    ta = 100.0 * t_correct / t_total
    va = 100.0 * v_correct / v_total
    tl = t_loss / len(train_loader)
    vl = v_loss / len(val_loader)

    train_losses.append(tl); val_losses.append(vl)
    train_accs.append(ta);   val_accs.append(va)
    scheduler.step()

    # Save the best model
    if va > best_acc:
        best_acc = va
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        marker = '  <-- NEW BEST SAVED!'
    else:
        marker = ''

    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'Train Loss: {tl:.4f}, Acc: {ta:.1f}% | '
          f'Val Loss: {vl:.4f}, Acc: {va:.1f}%{marker}')

print('\n' + '=' * 70)
print(f'Training DONE! Best Validation Accuracy: {best_acc:.1f}%')

In [ ]:
# STEP 6: Draw training progress charts
# Good signs: accuracy goes UP, loss goes DOWN
# Warning sign (overfitting): train accuracy much higher than val accuracy

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = range(1, NUM_EPOCHS + 1)

# Loss chart
ax1.plot(epochs_x, train_losses, 'b-o', label='Training Loss', markersize=3)
ax1.plot(epochs_x, val_losses, 'r-o', label='Validation Loss', markersize=3)
ax1.set_title('Loss Over Training\n(Lower = AI making fewer mistakes)', fontweight='bold')
ax1.set_xlabel('Epoch (study session number)')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy chart
ax2.plot(epochs_x, train_accs, 'b-o', label='Training Accuracy', markersize=3)
ax2.plot(epochs_x, val_accs, 'r-o', label='Validation Accuracy', markersize=3)
ax2.set_title('Accuracy Over Training\n(Higher = AI getting more answers right)', fontweight='bold')
ax2.set_xlabel('Epoch (study session number)')
ax2.set_ylabel('Accuracy %')
ax2.set_ylim(0, 100)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Smart Classroom CNN - Training Progress', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved! Final accuracy: Train={train_accs[-1]:.1f}%, Val={val_accs[-1]:.1f}%')

## STEP 7: Confusion Matrix

This shows WHERE the AI makes mistakes:
- **Diagonal** (top-left to bottom-right) = **correct** predictions
- **Off-diagonal** = mistakes (e.g. predicted HIGH but actually was MEDIUM)

Aim for numbers >80% on the diagonal!

In [ ]:
# Load the BEST model from training
model.load_state_dict(best_state)
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.numpy())

# Build confusion matrix
cm = confusion_matrix(all_true, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(8, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=[c.upper() for c in class_names],
            yticklabels=[c.upper() for c in class_names],
            cbar_kws={'label': 'Accuracy %'})
plt.title('Confusion Matrix - Where Does Our AI Get Confused?\n(Diagonal = correct, Off-diagonal = mistakes)',
          fontsize=12, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('AI Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDetailed Report:')
print('precision = How often the AI is right when it says a class')
print('recall    = How often the AI catches a class when it really is that class')
print('f1-score  = Balance of both (aim for >0.80)')
print()
print(classification_report(all_true, all_preds, target_names=[c.upper() for c in class_names]))

## STEP 8: Save & Export the Trained Model

We save in **ONNX format** - a universal file format that works inside Docker without needing PyTorch.

Think of it like converting a Word document to PDF - the PDF opens anywhere!

In [ ]:
from google.colab import files

# Load best model
model.load_state_dict(best_state)
model.eval()

# Save PyTorch format (.pth) - for retraining later
torch.save({
    'model_state_dict': best_state,
    'class_names': class_names,
    'image_size': IMAGE_SIZE,
    'best_val_accuracy': best_acc
}, 'classroom_occupancy_model.pth')
print('Saved: classroom_occupancy_model.pth (for retraining)')

# Export ONNX format (.onnx) - for Docker deployment
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
torch.onnx.export(
    model, dummy_input, 'classroom_occupancy.onnx',
    export_params=True, opset_version=11,
    input_names=['image_input'], output_names=['occupancy_scores'],
    dynamic_axes={'image_input': {0: 'batch'}, 'occupancy_scores': {0: 'batch'}}
)
print('Saved: classroom_occupancy.onnx (for Docker/Edge deployment)')

# Save class mapping config
with open('class_config.json', 'w') as f:
    json.dump({
        'class_names': class_names,
        'image_size': IMAGE_SIZE,
        'best_accuracy': best_acc,
        'model_info': 'Custom CNN trained from scratch - No pretrained weights used'
    }, f, indent=2)
print('Saved: class_config.json (class label mapping)')

print(f'\nBest Validation Accuracy: {best_acc:.1f}%')
print('\nDownloading all files to your computer...')

for fname in ['classroom_occupancy.onnx', 'classroom_occupancy_model.pth',
              'class_config.json', 'training_curves.png', 'confusion_matrix.png']:
    if os.path.exists(fname):
        files.download(fname)
        size_mb = os.path.getsize(fname) / 1024 / 1024
        print(f'  Downloaded: {fname} ({size_mb:.1f} MB)')

print('\nALL DONE! Next steps:')
print('  1. Copy classroom_occupancy.onnx  -->  edge_app/model/')
print('  2. Copy class_config.json         -->  edge_app/model/')
print('  3. Run: docker-compose up')
print('  4. Open: http://localhost:5000')